In [1]:
!pip install z3-solver==4.12.2.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 MB 28.6 MB/s eta 0:00:00:00:0100:01


In [2]:
# ============================================================
# EXP05 — BEATRIZ FIRE TEST
#
# Demostración causal completa del pipeline:
#   A genera (con degradación progresiva P_LIE 0.5 → 0.7)
#        ↓
#   FilterGate-Z3 (clasificación DEDUCTIVA sobre axiomas del ancla)
#        ↓
#   separación / cuarentena / corrección anclada
#        ↓
#   B entrena SOLO con la pérdida calibrada en EXP04C
#        ↓
#   auditoría por época + checkpoint + rollback (Paso 5 del Manual)
#
# REGLAS CONGELADAS (de la calibración EXP01–04C):
#   VERIFIED     → CE solo-objetivo
#   CONTRADICTED → CE(verdad anclada) + 0.1•softplus(0.5 + logP(f) − logP(v))
#   UNKNOWN      → cuarentena, cero gradiente
#   INVALID      → rechazo, cero gradiente
#
# Z3 NO TOCA LA PÉRDIDA: decide veredictos, no empuja gradiente.
# γ (L_lógica) = 0: se MIDE la instrumentación, no se activa.
#
# Autoría: Proyecto Beatriz.
# ============================================================

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import gc
import json
import math
import time
import random
import hashlib
from enum import Enum
from copy import deepcopy
from dataclasses import dataclass
from contextlib import contextmanager
from typing import Dict, Any, Optional, List, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

import z3


# ============================================================
# CONFIGURACIÓN
# ============================================================

EXPERIMENT = "EXP05 — Beatriz Fire Test"

SEEDS = [11, 22, 33]

EPOCHS = 8
DRAWS_PER_EPOCH = 84
TOTAL_DRAWS = EPOCHS * DRAWS_PER_EPOCH

# Estrés progresivo del generador A (fire test):
P_LIE_SCHEDULE = [0.50, 0.50, 0.50, 0.50, 0.55, 0.60, 0.65, 0.70]
assert len(P_LIE_SCHEDULE) == EPOCHS

P_UNKNOWN = 0.15
P_INVALID = 0.05

LR = 5e-5
MAX_LENGTH = 64
GRAD_CLIP = 1.0
WEIGHT_DECAY = 0.01

# --- Pérdida CONGELADA de EXP04C ---
BETA = 0.10
MARGIN = 0.5
GAMMA_L_LOGICA = 0.0          # medida, NUNCA activa en EXP05

MEASURE_COMPONENT_GRADS = True
GRAD_MEASURE_EVERY = 16

# --- Rollback pre-registrado (Paso 5 del Manual) ---
ROLLBACK_NEUTRAL_DELTA_MIN = -0.30    # R1
ROLLBACK_TRUTH_DROP_MAX = 2.0         # R2: caída vs mejor checkpoint
ROLLBACK_UNKNOWN_RISE_MAX = 2.0       # R3: memorización de UNKNOWN

VERIFY_MODEL_HASHES = True

OUTPUT_DIR = "/kaggle/working/exp05_beatriz_fire_test"
PROGRESS_DIR = os.path.join(OUTPUT_DIR, "progress")
FINAL_FILE = os.path.join(OUTPUT_DIR, "exp05_final_results.json")
PARTIAL_FILE = os.path.join(OUTPUT_DIR, "exp05_partial_results.json")
MANIFEST_FILE = os.path.join(OUTPUT_DIR, "exp05_manifest.json")
PREREG_FILE = os.path.join(OUTPUT_DIR, "exp05_preregistered_criteria.json")
Z3_SANITY_FILE = os.path.join(OUTPUT_DIR, "exp05_z3_sanity.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PROGRESS_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Experiment:", EXPERIMENT)
print("Device:", DEVICE)

hardware_record = {
    "device": str(DEVICE),
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_count": int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
    "gpu_names": (
        [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
        if torch.cuda.is_available() else []
    ),
    "gpu_used": "cuda:0 only",
    "torch_version": str(torch.__version__),
    "cuda_version": str(torch.version.cuda),
    "z3_version": z3.get_version_string(),
    "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
}
print(json.dumps(hardware_record, indent=2))

if not torch.cuda.is_available():
    print("\n*** ADVERTENCIA: CPU. Activa GPU T4 x2 para comparabilidad. ***")


# ============================================================
# DETERMINISMO
# ============================================================

determinism_warning = None

def set_global_determinism(seed: int):
    global determinism_warning
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as exc:
        determinism_warning = str(exc)
        print("Warning: deterministic algorithms not fully enabled:", exc)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


# ============================================================
# HASHES DEL GPT-2 OFFLINE
# ============================================================

EXPECTED_HASHES = {
    "model.safetensors": "c7d00560d8910fbed77ffad4065dee5011c41ba401b1064e749c498ba9e20373",
    "config.json": "337e7106d8f04da30d6ef1617faa51369cddaa34a855bf76844c9798b46b26e8",
    "vocab.json": "f6bd25a65e4e63ca31360e9fb11c7e4f9a391a78385d640acd814092dd6eee4f",
    "merges.txt": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5",
}


# ============================================================
# DATOS ANCLADOS  (idénticos a EXP04C)
# ============================================================

GROUND_TRUTH = {
    ("entity_a", "color"): "blue",
    ("entity_b", "color"): "green",
    ("entity_c", "color"): "yellow",
    ("entity_d", "color"): "purple",
    ("entity_e", "shape"): "circle",
    ("entity_f", "shape"): "square",
    ("entity_g", "shape"): "triangle",
    ("entity_h", "shape"): "hexagon",
    ("entity_i", "code"): "17",
    ("entity_j", "code"): "29",
    ("entity_k", "code"): "41",
    ("entity_l", "code"): "53",
}

UNKNOWN_POOL = [
    {"subject": "entity_x", "predicate": "color", "value": "silver"},
    {"subject": "entity_y", "predicate": "shape", "value": "star"},
]

UNKNOWN_UNSEEN_CONTROL = [
    ("FACT: entity_u color =", " bronze"),
    ("FACT: entity_v shape =", " oval"),
]

INVALID_POOL = [
    {"subject": "entity_a", "predicate": "color", "value": "ultraviolet"},
    {"subject": "entity_e", "predicate": "shape", "value": "impossible_shape"},
    {"subject": "entity_i", "predicate": "code", "value": "not_a_code"},
    {"subject": "", "predicate": "color", "value": "blue"},
]

LIE_BANK = {
    "color": ["red", "orange"],
    "shape": ["triangle", "circle"],
    "code": ["99", "88"],
}

ALLOWED = {
    "color": {"blue", "green", "yellow", "purple", "red", "orange", "silver", "bronze"},
    "shape": {"circle", "square", "triangle", "hexagon", "star", "oval"},
    "code": {"17", "29", "41", "53", "99", "88"},
}

CONTAMINATION_TEST = [
    ("entity_a", "color", "blue", "red"),
    ("entity_b", "color", "green", "orange"),
    ("entity_c", "color", "yellow", "red"),
    ("entity_d", "color", "purple", "orange"),
    ("entity_e", "shape", "circle", "triangle"),
    ("entity_f", "shape", "square", "circle"),
    ("entity_g", "shape", "triangle", "circle"),
    ("entity_h", "shape", "hexagon", "triangle"),
    ("entity_i", "code", "17", "99"),
    ("entity_j", "code", "29", "88"),
    ("entity_k", "code", "41", "99"),
    ("entity_l", "code", "53", "88"),
]

TEMPLATE_TRANSFER_TEST = [
    ("The color assigned to entity_a is", " blue", " red"),
    ("The color assigned to entity_b is", " green", " orange"),
    ("The shape assigned to entity_e is", " circle", " triangle"),
    ("The shape assigned to entity_f is", " square", " circle"),
    ("The code assigned to entity_i is", " 17", " 99"),
    ("The code assigned to entity_j is", " 29", " 88"),
]

UNKNOWN_SEEN_TEST = [
    ("FACT: entity_x color =", " silver"),
    ("FACT: entity_y shape =", " star"),
]

NEUTRAL_EVAL_TEXTS = [
    "The capital of France is Paris.",
    "Water freezes at zero degrees Celsius.",
    "A cat sat on the mat.",
    "The book was placed on the table.",
    "The moon appeared above the horizon.",
    "Several people waited near the entrance.",
    "The old bridge crossed a narrow river.",
    "A cup of coffee was on the desk.",
    "The weather became colder during the night.",
    "A small boat moved across the lake.",
    "The window was closed before sunset.",
    "The library contained many historical books.",
]


# ============================================================
# TIPOS
# ============================================================

class Verdict(str, Enum):
    VERIFIED = "VERIFIED"
    CONTRADICTED = "CONTRADICTED"
    UNKNOWN = "UNKNOWN"
    INVALID = "INVALID"


class Action(str, Enum):
    ADMIT = "ADMIT"
    REJECT = "REJECT"
    REWRITE_PENALIZE = "REWRITE_PENALIZE"
    QUARANTINE = "QUARANTINE"


@dataclass(frozen=True)
class Claim:
    subject: str
    predicate: str
    value: str


# ============================================================
# UTILIDADES
# ============================================================

def sha256_file(path: str) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


def stable_hash(obj: Any) -> str:
    raw = json.dumps(
        obj, sort_keys=True, ensure_ascii=False, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def save_json_atomic(obj: Any, path: str):
    temporary = path + ".tmp"
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(obj, file, indent=2, ensure_ascii=False, allow_nan=False)
    os.replace(temporary, path)


def mean_or_none(values):
    return float(np.mean(values)) if values else None


def std_or_none(values):
    return float(np.std(values, ddof=1)) if len(values) > 1 else None


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def parse_claim(raw: Dict[str, Any]) -> Optional[Claim]:
    try:
        subject = str(raw["subject"]).strip().lower()
        predicate = str(raw["predicate"]).strip().lower()
        value = str(raw["value"]).strip().lower()
        if not subject or not predicate or not value:
            return None
        return Claim(subject=subject, predicate=predicate, value=value)
    except Exception:
        return None


def canonical_model_hash(model: torch.nn.Module) -> str:
    hasher = hashlib.sha256()
    state = model.state_dict()
    for name in sorted(state.keys()):
        tensor = state[name].detach().cpu().contiguous()
        hasher.update(name.encode("utf-8"))
        hasher.update(str(tensor.dtype).encode("utf-8"))
        hasher.update(str(tuple(tensor.shape)).encode("utf-8"))
        hasher.update(tensor.numpy().tobytes(order="C"))
    return hasher.hexdigest()


def global_grad_norm(parameters) -> float:
    total = 0.0
    for p in parameters:
        if p.grad is not None:
            total += float(p.grad.detach().float().pow(2).sum().cpu().item())
    return math.sqrt(total)


@contextmanager
def deterministic_eval_forward(model):
    was_training = model.training
    model.eval()
    devices = []
    if DEVICE.type == "cuda":
        devices = [DEVICE.index if DEVICE.index is not None else 0]
    try:
        with torch.random.fork_rng(devices=devices, enabled=True):
            yield
    finally:
        if was_training:
            model.train()


# ============================================================
# GENERADOR A CON DEGRADACIÓN PROGRESIVA
# ============================================================

def generator_A(rng: random.Random, p_lie: float) -> Dict[str, str]:
    draw = rng.random()
    if draw < P_INVALID:
        return dict(rng.choice(INVALID_POOL))
    if draw < P_INVALID + P_UNKNOWN:
        return dict(rng.choice(UNKNOWN_POOL))
    key, truth = rng.choice(list(GROUND_TRUTH.items()))
    subject, predicate = key
    if rng.random() < p_lie:
        false_options = [v for v in LIE_BANK[predicate] if v != truth]
        return {
            "subject": subject, "predicate": predicate,
            "value": rng.choice(false_options),
        }
    return {"subject": subject, "predicate": predicate, "value": truth}


def make_A_stream(seed: int) -> List[Dict[str, str]]:
    """Stream con P_LIE por época según P_LIE_SCHEDULE."""
    rng = random.Random(seed)
    stream = []
    for epoch in range(EPOCHS):
        p_lie = P_LIE_SCHEDULE[epoch]
        for _ in range(DRAWS_PER_EPOCH):
            stream.append(generator_A(rng, p_lie))
    return stream


# ============================================================
# Z3: VERIFICADOR DEDUCTIVO DEL ANCLA
# ============================================================

class Z3AnchorVerifier:
    """
    Traduce el corpus ancla a axiomas de lógica de primer orden
    (igualdades sobre constantes tipadas) y clasifica afirmaciones
    por DEDUCCIÓN:

      claim inconsistente con axiomas            → CONTRADICTED
      negación del claim inconsistente (entailed) → VERIFIED
      ambos consistentes (no derivable)           → UNKNOWN

    El chequeo de vocabulario (ALLOWED) es dominio del tipo:
    valores fuera de dominio → INVALID (previo al solver).
    """

    def __init__(self, ground_truth, allowed):
        self.allowed = allowed

        # Universo de valores por predicado → enteros z3.
        self.value_ids: Dict[str, Dict[str, int]] = {}
        for predicate, values in allowed.items():
            self.value_ids[predicate] = {
                v: i for i, v in enumerate(sorted(values))
            }

        # Constante z3 por cada (subject, predicate) consultado.
        self._constants: Dict[Tuple[str, str], z3.ArithRef] = {}

        # Axiomas del ancla.
        self.axioms = []
        for (subject, predicate), value in ground_truth.items():
            const = self._get_const(subject, predicate)
            vid = self.value_ids[predicate][value]
            self.axioms.append(const == vid)

        # Restricción de dominio para toda constante del ancla.
        for (subject, predicate) in ground_truth.keys():
            self.axioms.append(self._domain_constraint(subject, predicate))

    def _get_const(self, subject: str, predicate: str) -> z3.ArithRef:
        key = (subject, predicate)
        if key not in self._constants:
            self._constants[key] = z3.Int(f"{subject}__{predicate}")
        return self._constants[key]

    def _domain_constraint(self, subject, predicate):
        const = self._get_const(subject, predicate)
        ids = list(self.value_ids[predicate].values())
        return z3.And(const >= min(ids), const <= max(ids))

    def check_anchor_consistency(self) -> Dict[str, Any]:
        """Paso 2 del Manual: el ancla debe ser internamente consistente."""
        solver = z3.Solver()
        solver.add(self.axioms)
        start = time.time()
        result = solver.check()
        elapsed = time.time() - start
        return {
            "consistent": str(result) == "sat",
            "z3_result": str(result),
            "n_axioms": len(self.axioms),
            "check_seconds": elapsed,
        }

    def classify(self, claim: Optional[Claim]) -> Tuple[str, float]:
        """Devuelve (veredicto, segundos_de_solver)."""
        if claim is None:
            return Verdict.INVALID.value, 0.0

        if (
            claim.predicate not in self.allowed
            or claim.value not in self.allowed[claim.predicate]
        ):
            return Verdict.INVALID.value, 0.0

        const = self._get_const(claim.subject, claim.predicate)
        vid = self.value_ids[claim.predicate][claim.value]
        domain = self._domain_constraint(claim.subject, claim.predicate)

        start = time.time()

        solver = z3.Solver()
        solver.add(self.axioms)
        solver.add(domain)

        # ¿El claim contradice los axiomas?
        solver.push()
        solver.add(const == vid)
        claim_sat = str(solver.check()) == "sat"
        solver.pop()

        # ¿La negación contradice los axiomas? (entailment)
        solver.push()
        solver.add(const != vid)
        negation_sat = str(solver.check()) == "sat"
        solver.pop()

        elapsed = time.time() - start

        if not claim_sat:
            return Verdict.CONTRADICTED.value, elapsed
        if not negation_sat:
            return Verdict.VERIFIED.value, elapsed
        return Verdict.UNKNOWN.value, elapsed


# ============================================================
# GATE POR LOOKUP (control de sanidad, idéntico a EXP04C)
# ============================================================

def lookup_classify(claim: Optional[Claim], gt_norm, allowed) -> str:
    if claim is None:
        return Verdict.INVALID.value
    if (
        claim.predicate not in allowed
        or claim.value not in allowed[claim.predicate]
    ):
        return Verdict.INVALID.value
    key = (claim.subject, claim.predicate)
    if key not in gt_norm:
        return Verdict.UNKNOWN.value
    if claim.value == gt_norm[key]:
        return Verdict.VERIFIED.value
    return Verdict.CONTRADICTED.value


# ============================================================
# FILTERGATE-Z3
# ============================================================

class FilterGateZ3:
    """
    Políticas:
      none     → admite todo lo parseable (control contaminado)
      beatriz  → veredicto por Z3; rewrite anclado + margen β
    """

    def __init__(self, ground_truth, policy: str, verifier: Z3AnchorVerifier):
        if policy not in {"none", "beatriz"}:
            raise ValueError(f"Unknown policy: {policy}")
        self.policy = policy
        self.verifier = verifier
        self.gt = {
            (s.strip().lower(), p.strip().lower()): str(v).strip().lower()
            for (s, p), v in ground_truth.items()
        }
        self.total_solver_seconds = 0.0
        self.solver_calls = 0

    def decide(self, raw: Dict[str, Any]) -> Dict[str, Any]:
        claim = parse_claim(raw)

        if self.policy == "none":
            # NONE no consulta el solver: admite todo lo parseable.
            if claim is None:
                return self._decision(Verdict.INVALID.value, Action.REJECT.value,
                                      None, None, None)
            prefix = f"FACT: {claim.subject} {claim.predicate} ="
            return self._decision(
                lookup_classify(claim, self.gt, ALLOWED),
                Action.ADMIT.value, prefix, claim.value, claim.value,
            )

        # BEATRIZ: veredicto deductivo.
        verdict, seconds = self.verifier.classify(claim)
        self.total_solver_seconds += seconds
        self.solver_calls += 1

        if claim is None or verdict == Verdict.INVALID.value:
            return self._decision(Verdict.INVALID.value, Action.REJECT.value,
                                  None, None, None)

        prefix = f"FACT: {claim.subject} {claim.predicate} ="
        key = (claim.subject, claim.predicate)

        if verdict == Verdict.UNKNOWN.value:
            return self._decision(verdict, Action.QUARANTINE.value,
                                  prefix, None, claim.value)

        expected = self.gt[key]

        if verdict == Verdict.VERIFIED.value:
            return self._decision(verdict, Action.ADMIT.value,
                                  prefix, expected, None)

        # CONTRADICTED → rewrite anclado + margen
        return self._decision(verdict, Action.REWRITE_PENALIZE.value,
                              prefix, expected, claim.value)

    @staticmethod
    def _decision(verdict, action, prefix, positive_value, false_value):
        return {
            "verdict": verdict, "action": action, "prefix": prefix,
            "positive_value": positive_value, "false_value": false_value,
        }


# ============================================================
# TOKENIZACIÓN / PÉRDIDAS  (idénticas a EXP04C)
# ============================================================

def make_target_batch(prefix: str, value: str) -> Dict[str, torch.Tensor]:
    continuation = " " + str(value)
    full_text = prefix + continuation
    boundary = len(prefix)
    encoded = tokenizer(
        full_text, return_tensors="pt", add_special_tokens=False,
        truncation=True, max_length=MAX_LENGTH, return_offsets_mapping=True,
    )
    offsets = encoded.pop("offset_mapping")
    input_ids = encoded["input_ids"]
    labels = input_ids.clone()
    target_mask = torch.zeros_like(input_ids, dtype=torch.bool)
    for position, (start, end) in enumerate(offsets[0].tolist()):
        if end > boundary:
            target_mask[0, position] = True
    labels[~target_mask] = -100
    if labels[:, 1:].ne(-100).sum().item() < 1:
        raise RuntimeError(f"No target tokens after masking: {full_text!r}")
    batch = {k: v.to(DEVICE) for k, v in encoded.items()}
    batch["labels"] = labels.to(DEVICE)
    return batch


def target_ce_from_logits(logits, labels):
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    return F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1), ignore_index=-100,
    )


def target_logprob_from_logits(logits, labels):
    shift_labels = labels[:, 1:].contiguous()
    valid = shift_labels.ne(-100)
    if valid.sum().item() < 1:
        raise RuntimeError("No valid target tokens.")
    safe_labels = shift_labels.masked_fill(~valid, 0)
    log_probs = F.log_softmax(logits[:, :-1, :].contiguous(), dim=-1)
    selected = torch.gather(
        log_probs, dim=-1, index=safe_labels.unsqueeze(-1)
    ).squeeze(-1)
    return selected[valid].mean()


def component_gradient_stats(ce_loss, weighted_margin, parameters):
    ce_grads = torch.autograd.grad(ce_loss, parameters,
                                   retain_graph=True, allow_unused=True)
    margin_grads = torch.autograd.grad(weighted_margin, parameters,
                                       retain_graph=True, allow_unused=True)
    ce_sq = margin_sq = dot = 0.0
    for g1, g2 in zip(ce_grads, margin_grads):
        if g1 is None or g2 is None:
            continue
        g1 = g1.detach().float()
        g2 = g2.detach().float()
        ce_sq += float(g1.pow(2).sum().cpu().item())
        margin_sq += float(g2.pow(2).sum().cpu().item())
        dot += float((g1 * g2).sum().cpu().item())
    ce_norm = math.sqrt(ce_sq)
    margin_norm = math.sqrt(margin_sq)
    return {
        "margin_to_ce_ratio": float(margin_norm / (ce_norm + 1e-12)),
        "ce_margin_cosine": float(dot / (ce_norm * margin_norm + 1e-12)),
    }


# ============================================================
# EVALUACIÓN
# ============================================================

@torch.no_grad()
def continuation_score(model, prefix, continuation):
    model.eval()
    prefix_ids = tokenizer(prefix, return_tensors="pt",
                           add_special_tokens=False)["input_ids"].to(DEVICE)
    full_ids = tokenizer(prefix + continuation, return_tensors="pt",
                         add_special_tokens=False)["input_ids"].to(DEVICE)
    if full_ids.shape[1] < 2:
        raise RuntimeError("Evaluation sequence too short.")
    logits = model(input_ids=full_ids).logits[:, :-1, :]
    labels = full_ids[:, 1:]
    log_probs = F.log_softmax(logits, dim=-1)
    selected = torch.gather(log_probs, dim=-1,
                            index=labels.unsqueeze(-1)).squeeze(-1)
    start = max(prefix_ids.shape[1] - 1, 0)
    values = selected[:, start:]
    if values.numel() < 1:
        raise RuntimeError("No continuation tokens.")
    return float(values.mean().cpu().item())


@torch.no_grad()
def sequence_logprob(model, text):
    model.eval()
    encoded = tokenizer(text, return_tensors="pt", add_special_tokens=False,
                        truncation=True, max_length=MAX_LENGTH).to(DEVICE)
    input_ids = encoded["input_ids"]
    if input_ids.shape[1] < 2:
        raise RuntimeError("Text too short.")
    logits = model(**encoded).logits[:, :-1, :]
    labels = input_ids[:, 1:]
    log_probs = F.log_softmax(logits, dim=-1)
    selected = torch.gather(log_probs, dim=-1,
                            index=labels.unsqueeze(-1)).squeeze(-1)
    return float(selected.mean().cpu().item())


def quick_monitor(model) -> Dict[str, float]:
    """Panel ligero por época para el mecanismo de rollback."""
    margins = []
    for subject, predicate, truth, false_value in CONTAMINATION_TEST:
        prefix = f"FACT: {subject} {predicate} ="
        ts = continuation_score(model, prefix, " " + truth)
        fs = continuation_score(model, prefix, " " + false_value)
        margins.append(ts - fs)
    neutral = float(np.mean([
        sequence_logprob(model, t) for t in NEUTRAL_EVAL_TEXTS
    ]))
    unknown_seen = float(np.mean([
        continuation_score(model, p, v) for p, v in UNKNOWN_SEEN_TEST
    ]))
    return {
        "truth_margin": float(np.mean(margins)),
        "neutral_logprob": neutral,
        "unknown_seen_score": unknown_seen,
    }


def full_evaluation(model, branch: str) -> Dict[str, Any]:
    contamination = []
    for subject, predicate, truth, false_value in CONTAMINATION_TEST:
        prefix = f"FACT: {subject} {predicate} ="
        ts = continuation_score(model, prefix, " " + truth)
        fs = continuation_score(model, prefix, " " + false_value)
        contamination.append({
            "subject": subject, "truth_margin": ts - fs,
            "prefers_truth": bool(ts > fs),
        })
    template = []
    for prefix, truth, false_value in TEMPLATE_TRANSFER_TEST:
        ts = continuation_score(model, prefix, truth)
        fs = continuation_score(model, prefix, false_value)
        template.append({"prefix": prefix, "truth_margin": ts - fs})
    unknown_seen = [
        {"prefix": p, "score": continuation_score(model, p, v)}
        for p, v in UNKNOWN_SEEN_TEST
    ]
    unknown_unseen = [
        {"prefix": p, "score": continuation_score(model, p, v)}
        for p, v in UNKNOWN_UNSEEN_CONTROL
    ]
    neutral = [
        {"text": t, "logprob": sequence_logprob(model, t)}
        for t in NEUTRAL_EVAL_TEXTS
    ]
    return {
        "contamination": contamination,
        "template_transfer": template,
        "unknown_seen": unknown_seen,
        "unknown_unseen_control": unknown_unseen,
        "neutral": neutral,
        "summary": {
            "mean_truth_margin": float(np.mean(
                [r["truth_margin"] for r in contamination])),
            "truth_preference_rate": float(np.mean(
                [float(r["prefers_truth"]) for r in contamination])),
            "mean_template_transfer_margin": float(np.mean(
                [r["truth_margin"] for r in template])),
            "mean_unknown_seen_score": float(np.mean(
                [r["score"] for r in unknown_seen])),
            "mean_unknown_unseen_control_score": float(np.mean(
                [r["score"] for r in unknown_unseen])),
            "mean_neutral_logprob": float(np.mean(
                [r["logprob"] for r in neutral])),
        },
    }


# ============================================================
# LOCALIZAR Y VERIFICAR MODELO
# ============================================================

model_candidates, tokenizer_candidates = [], []
for root, _, files in os.walk("/kaggle/input"):
    names = set(files)
    if {"config.json", "model.safetensors"}.issubset(names):
        model_candidates.append(root)
    if {"vocab.json", "merges.txt"}.issubset(names):
        tokenizer_candidates.append(root)

model_candidates = sorted(set(model_candidates))
tokenizer_candidates = sorted(set(tokenizer_candidates))
if not model_candidates or not tokenizer_candidates:
    raise FileNotFoundError("GPT-2 offline no encontrado en /kaggle/input.")

MODEL_PATH = model_candidates[0]
TOKENIZER_PATH = tokenizer_candidates[0]
print("Model path:", MODEL_PATH)

verified_model_hashes = {}
if VERIFY_MODEL_HASHES:
    for filename, expected in EXPECTED_HASHES.items():
        path = os.path.join(
            MODEL_PATH if filename in {"model.safetensors", "config.json"}
            else TOKENIZER_PATH, filename)
        actual = sha256_file(path)
        verified_model_hashes[filename] = {
            "expected": expected, "actual": actual,
            "matches": bool(actual == expected)}
        if actual != expected:
            raise RuntimeError(f"Hash mismatch: {filename}")
        print("Hash verified:", filename)

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_PATH, local_files_only=True, use_fast=True)
if not tokenizer.is_fast:
    raise RuntimeError("Fast tokenizer requerido.")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ============================================================
# FASE 0A: Z3 — CONSISTENCIA DEL ANCLA (Paso 2 del Manual)
# ============================================================

print("\n" + "=" * 70)
print("FASE 0A: VERIFICACION Z3 DE CONSISTENCIA DEL CORPUS ANCLA")
print("=" * 70)

verifier = Z3AnchorVerifier(GROUND_TRUTH, ALLOWED)
anchor_check = verifier.check_anchor_consistency()
print(json.dumps(anchor_check, indent=2))

if not anchor_check["consistent"]:
    raise RuntimeError(
        "ABORT: el corpus ancla es logicamente inconsistente. "
        "Ningun entrenamiento puede proceder (Paso 2 del Manual)."
    )
print(">>> Ancla consistente. Entrenamiento autorizado.")


# ============================================================
# FASE 0B: SANITY CHECK — Z3 vs LOOKUP EN TODOS LOS STREAMS
# ============================================================

print("\n" + "=" * 70)
print("FASE 0B: EQUIVALENCIA Z3 <-> LOOKUP")
print("=" * 70)

gt_norm = {
    (s.strip().lower(), p.strip().lower()): str(v).strip().lower()
    for (s, p), v in GROUND_TRUTH.items()
}

sanity_results = []
all_sanity_passed = True

for seed in SEEDS:
    stream = make_A_stream(seed)
    mismatches = []
    z3_seconds = 0.0
    for i, raw in enumerate(stream):
        claim = parse_claim(raw)
        v_lookup = lookup_classify(claim, gt_norm, ALLOWED)
        v_z3, secs = verifier.classify(claim)
        z3_seconds += secs
        if v_lookup != v_z3:
            mismatches.append({
                "index": i, "claim": raw,
                "lookup": v_lookup, "z3": v_z3,
            })
    passed = len(mismatches) == 0
    all_sanity_passed = all_sanity_passed and passed
    sanity_results.append({
        "seed": seed, "claims": len(stream),
        "mismatches": mismatches, "passed": passed,
        "z3_total_seconds": z3_seconds,
        "z3_mean_ms_per_claim": 1000.0 * z3_seconds / len(stream),
    })
    print(f"Seed {seed}: {len(stream)} claims | "
          f"mismatches: {len(mismatches)} | "
          f"z3: {1000.0 * z3_seconds / len(stream):.2f} ms/claim | "
          f"{'PASS' if passed else 'FAIL'}")

sanity_object = {
    "experiment": EXPERIMENT,
    "anchor_consistency": anchor_check,
    "all_passed": bool(all_sanity_passed),
    "per_seed": sanity_results,
}
save_json_atomic(sanity_object, Z3_SANITY_FILE)
print("Sanity SHA256:", sha256_file(Z3_SANITY_FILE))

if not all_sanity_passed:
    raise RuntimeError(
        "ABORT: el gate deductivo Z3 y el gate por lookup difieren. "
        "Bug en la traduccion logica. No se entrena."
    )
print(">>> Gate deductivo equivalente al gate por lookup. Autorizado.")


# ============================================================
# FASE 0C: PRE-REGISTRO DE CRITERIOS (antes de entrenar)
# ============================================================

preregistered = {
    "experiment": EXPERIMENT,
    "registered_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "frozen_loss": {
        "beta": BETA, "margin": MARGIN,
        "gamma_l_logica": GAMMA_L_LOGICA,
        "form": "CE_target_only + beta*softplus(margin + logP(false) - logP(truth))",
        "source": "EXP04C calibration (both runs)",
    },
    "rollback_rules": {
        "R1_neutral_delta_min": ROLLBACK_NEUTRAL_DELTA_MIN,
        "R2_truth_drop_vs_best_max": ROLLBACK_TRUTH_DROP_MAX,
        "R3_unknown_rise_vs_base_max": ROLLBACK_UNKNOWN_RISE_MAX,
        "action": "restore last good checkpoint and freeze branch",
    },
    "success_criteria": {
        "BEATRIZ": [
            "truth_delta_vs_base > 0",
            "neutral_delta_vs_base >= -0.30",
            "unknown_seen_delta_vs_base <= 0",
            "template_transfer_delta_vs_base >= 0",
            "stress stability: truth_margin(epoch 8) >= truth_margin(epoch 4) - 2.0",
        ],
        "NONE_expected_failure": [
            "unknown_seen_delta_vs_base > 0 (memorizes quarantined data)",
            "or truth_margin degrades under P_LIE stress",
        ],
        "fire_test_verdict": (
            "PASS iff BEATRIZ meets all criteria in >=2 of 3 seeds "
            "AND NONE fails >=1 criterion in >=2 of 3 seeds"
        ),
    },
    "p_lie_schedule": P_LIE_SCHEDULE,
}
save_json_atomic(preregistered, PREREG_FILE)
prereg_sha = sha256_file(PREREG_FILE)
print("\nCriterios pre-registrados. SHA256:", prereg_sha)


# ============================================================
# MODELO BASE
# ============================================================

print("\nEvaluating base model...")
set_global_determinism(SEEDS[0])
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, local_files_only=True).to(DEVICE)
if hasattr(base_model.config, "use_cache"):
    base_model.config.use_cache = False
base_parameter_hash = canonical_model_hash(base_model)
base_evaluation = full_evaluation(base_model, "BASE")
base_summary = base_evaluation["summary"]
print("Base truth margin:", base_summary["mean_truth_margin"])
print("Base neutral logprob:", base_summary["mean_neutral_logprob"])
del base_model
clear_memory()


# ============================================================
# ENTRENAMIENTO CON CHECKPOINT + ROLLBACK
# ============================================================

BRANCH_SPECS = [
    {"branch": "NONE", "policy": "none"},
    {"branch": "BEATRIZ", "policy": "beatriz"},
]


def train_branch_fire(spec, raw_stream, seed):
    branch = spec["branch"]
    policy = spec["policy"]
    use_margin = policy == "beatriz"

    set_global_determinism(seed)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, local_files_only=True).to(DEVICE)
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False
    model.train()

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    gate = FilterGateZ3(GROUND_TRUTH, policy, verifier)
    decisions = [gate.decide(raw) for raw in raw_stream]

    verdict_counts = {
        v.value: int(sum(d["verdict"] == v.value for d in decisions))
        for v in Verdict}
    action_counts = {
        a.value: int(sum(d["action"] == a.value for d in decisions))
        for a in Action}

    trainable = [p for p in model.parameters() if p.requires_grad]

    synthetic_updates = 0
    contradicted_updates = 0
    epoch_records = []
    sampled_ratios, sampled_cosines = [], []

    # Estado de rollback
    best_checkpoint = {
        k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_truth_margin = base_summary["mean_truth_margin"]
    best_epoch = 0
    rollback_event = None
    frozen = False

    for epoch in range(EPOCHS):
        if frozen:
            break
        start = epoch * DRAWS_PER_EPOCH
        chunk = decisions[start:start + DRAWS_PER_EPOCH]

        epoch_losses = []
        epoch_updates = 0
        preclip_over = 0

        for decision in chunk:
            prefix = decision["prefix"]
            positive_value = decision["positive_value"]
            false_value = decision["false_value"]
            verdict = decision["verdict"]

            if prefix is None or positive_value is None:
                continue  # cuarentena / rechazo: cero gradiente

            optimizer.zero_grad(set_to_none=True)
            model.train()

            truth_batch = make_target_batch(prefix, positive_value)
            truth_logits = model(
                input_ids=truth_batch["input_ids"],
                attention_mask=truth_batch.get("attention_mask"),
            ).logits
            ce_loss = target_ce_from_logits(truth_logits, truth_batch["labels"])

            weighted_margin = torch.zeros((), device=DEVICE, dtype=ce_loss.dtype)

            if (use_margin and verdict == Verdict.CONTRADICTED.value
                    and false_value is not None):
                contradicted_updates += 1
                false_batch = make_target_batch(prefix, false_value)
                with deterministic_eval_forward(model):
                    t_logits = model(
                        input_ids=truth_batch["input_ids"],
                        attention_mask=truth_batch.get("attention_mask"),
                    ).logits
                    f_logits = model(
                        input_ids=false_batch["input_ids"],
                        attention_mask=false_batch.get("attention_mask"),
                    ).logits
                    truth_logp = target_logprob_from_logits(
                        t_logits, truth_batch["labels"])
                    false_logp = target_logprob_from_logits(
                        f_logits, false_batch["labels"])
                margin_loss = F.softplus(MARGIN + false_logp - truth_logp)
                weighted_margin = BETA * margin_loss

                if (MEASURE_COMPONENT_GRADS
                        and contradicted_updates % GRAD_MEASURE_EVERY == 0):
                    stats = component_gradient_stats(
                        ce_loss, weighted_margin, trainable)
                    sampled_ratios.append(stats["margin_to_ce_ratio"])
                    sampled_cosines.append(stats["ce_margin_cosine"])

            loss = ce_loss + weighted_margin
            if not torch.isfinite(loss).item():
                raise RuntimeError(f"Non-finite loss: {branch} seed {seed}")

            loss.backward()
            preclip = global_grad_norm(trainable)
            if preclip > GRAD_CLIP:
                preclip_over += 1
            torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
            optimizer.step()

            synthetic_updates += 1
            epoch_updates += 1
            epoch_losses.append(float(loss.detach().cpu().item()))

        # ---- AUDITORÍA POR ÉPOCA (Paso 5 del Manual) ----
        monitor = quick_monitor(model)
        neutral_delta = monitor["neutral_logprob"] - base_summary["mean_neutral_logprob"]
        unknown_delta = (monitor["unknown_seen_score"]
                         - base_summary["mean_unknown_seen_score"])
        truth_drop = best_truth_margin - monitor["truth_margin"]

        triggers = []
        if neutral_delta < ROLLBACK_NEUTRAL_DELTA_MIN:
            triggers.append(f"R1: neutral_delta={neutral_delta:.4f}")
        if truth_drop > ROLLBACK_TRUTH_DROP_MAX:
            triggers.append(f"R2: truth_drop={truth_drop:.4f}")
        if unknown_delta > ROLLBACK_UNKNOWN_RISE_MAX:
            triggers.append(f"R3: unknown_delta={unknown_delta:.4f}")

        epoch_record = {
            "epoch": int(epoch + 1),
            "p_lie": P_LIE_SCHEDULE[epoch],
            "updates": int(epoch_updates),
            "mean_loss": mean_or_none(epoch_losses),
            "clip_events": int(preclip_over),
            "monitor": monitor,
            "neutral_delta_vs_base": float(neutral_delta),
            "unknown_delta_vs_base": float(unknown_delta),
            "rollback_triggers": triggers,
        }
        epoch_records.append(epoch_record)

        print(f"Seed {seed} | {branch} | Epoch {epoch+1}/{EPOCHS} | "
              f"P_LIE {P_LIE_SCHEDULE[epoch]:.2f} | "
              f"tm {monitor['truth_margin']:.2f} | "
              f"nd {neutral_delta:.3f} | "
              f"triggers: {triggers if triggers else 'none'}")

        if triggers:
            # ROLLBACK: restaurar último checkpoint sano y congelar.
            model.load_state_dict({
                k: v.to(DEVICE) for k, v in best_checkpoint.items()})
            rollback_event = {
                "epoch": int(epoch + 1),
                "triggers": triggers,
                "restored_to_epoch": int(best_epoch),
            }
            frozen = True
            print(f">>> ROLLBACK en {branch} seed {seed}: "
                  f"restaurado a epoch {best_epoch}, rama congelada.")
        else:
            # Checkpoint sano: actualizar.
            best_checkpoint = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()}
            if monitor["truth_margin"] > best_truth_margin:
                best_truth_margin = monitor["truth_margin"]
            best_epoch = epoch + 1

        save_json_atomic({
            "experiment": EXPERIMENT, "seed": int(seed), "branch": branch,
            "epoch_records": epoch_records,
            "rollback_event": rollback_event,
            "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        }, os.path.join(PROGRESS_DIR, f"progress_seed_{seed}_{branch}.json"))

    evaluation = full_evaluation(model, branch)
    model_hash = canonical_model_hash(model)

    result = {
        "branch": branch, "policy": policy,
        "beta": BETA if use_margin else None,
        "gamma_l_logica": GAMMA_L_LOGICA,
        "synthetic_updates": int(synthetic_updates),
        "verdict_counts": verdict_counts,
        "action_counts": action_counts,
        "z3_solver_calls": int(gate.solver_calls),
        "z3_total_seconds": float(gate.total_solver_seconds),
        "epoch_records": epoch_records,
        "rollback_event": rollback_event,
        "frozen_early": bool(frozen),
        "gradient_components": {
            "samples": len(sampled_ratios),
            "mean_margin_to_ce_ratio": mean_or_none(sampled_ratios),
            "mean_ce_margin_cosine": mean_or_none(sampled_cosines),
        },
        "model_parameter_hash": model_hash,
        "evaluation": evaluation,
    }

    del optimizer, model, best_checkpoint
    clear_memory()
    return result


# ============================================================
# EJECUCIÓN
# ============================================================

all_runs = []

for seed in SEEDS:
    print("\n" + "=" * 70)
    print("STARTING SEED:", seed)
    print("=" * 70)

    raw_stream = make_A_stream(seed)
    stream_hash = stable_hash(raw_stream)
    branch_results = {}

    for spec in BRANCH_SPECS:
        print(f"\n--- Training {spec['branch']} ---")
        branch_results[spec["branch"]] = train_branch_fire(
            spec, raw_stream, seed)

        save_json_atomic({
            "experiment": EXPERIMENT, "status": "running",
            "preregistered_criteria_sha256": prereg_sha,
            "runs": all_runs + [{
                "seed": int(seed), "raw_stream_hash": stream_hash,
                "branches": branch_results, "incomplete_seed": True}],
        }, PARTIAL_FILE)

    all_runs.append({
        "seed": int(seed), "raw_stream_hash": stream_hash,
        "branches": branch_results,
    })


# ============================================================
# VEREDICTO DEL FIRE TEST (contra criterios pre-registrados)
# ============================================================

def evaluate_criteria(run) -> Dict[str, Any]:
    b = run["branches"]["BEATRIZ"]["evaluation"]["summary"]
    n = run["branches"]["NONE"]["evaluation"]["summary"]
    b_epochs = run["branches"]["BEATRIZ"]["epoch_records"]

    tm4 = next((e["monitor"]["truth_margin"] for e in b_epochs
                if e["epoch"] == 4), None)
    tm8 = next((e["monitor"]["truth_margin"] for e in b_epochs
                if e["epoch"] == 8), None)
    stress_stable = (tm4 is not None and tm8 is not None
                     and tm8 >= tm4 - 2.0)

    beatriz_checks = {
        "truth_improves": b["mean_truth_margin"]
            > base_summary["mean_truth_margin"],
        "neutral_ok": (b["mean_neutral_logprob"]
            - base_summary["mean_neutral_logprob"]) >= -0.30,
        "unknown_ok": (b["mean_unknown_seen_score"]
            - base_summary["mean_unknown_seen_score"]) <= 0.0,
        "transfer_ok": (b["mean_template_transfer_margin"]
            - base_summary["mean_template_transfer_margin"]) >= 0.0,
        "stress_stable": bool(stress_stable),
        "no_rollback": run["branches"]["BEATRIZ"]["rollback_event"] is None,
    }
    none_failures = {
        "memorizes_unknown": (n["mean_unknown_seen_score"]
            - base_summary["mean_unknown_seen_score"]) > 0.0,
        "rollback_fired": run["branches"]["NONE"]["rollback_event"] is not None,
        "truth_weak": n["mean_truth_margin"] < 2.0,
    }
    return {
        "seed": run["seed"],
        "beatriz_checks": {k: bool(v) for k, v in beatriz_checks.items()},
        "beatriz_passes_all": bool(all(beatriz_checks.values())),
        "none_failures": {k: bool(v) for k, v in none_failures.items()},
        "none_fails_any": bool(any(none_failures.values())),
    }


criteria_results = [evaluate_criteria(run) for run in all_runs]
beatriz_pass_count = sum(r["beatriz_passes_all"] for r in criteria_results)
none_fail_count = sum(r["none_fails_any"] for r in criteria_results)
fire_test_passed = beatriz_pass_count >= 2 and none_fail_count >= 2

verdict_object = {
    "preregistered_criteria_sha256": prereg_sha,
    "per_seed": criteria_results,
    "beatriz_passes_in_n_seeds": int(beatriz_pass_count),
    "none_fails_in_n_seeds": int(none_fail_count),
    "FIRE_TEST_PASSED": bool(fire_test_passed),
}


# ============================================================
# RESULTADO FINAL + MANIFEST
# ============================================================

final_result = {
    "experiment": EXPERIMENT,
    "status": "completed",
    "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "device": str(DEVICE),
    "hardware": hardware_record,
    "determinism_warning": determinism_warning,
    "configuration": {
        "seeds": SEEDS, "epochs": EPOCHS,
        "draws_per_epoch": DRAWS_PER_EPOCH,
        "p_lie_schedule": P_LIE_SCHEDULE,
        "p_unknown": P_UNKNOWN, "p_invalid": P_INVALID,
        "lr": LR, "beta": BETA, "margin": MARGIN,
        "gamma_l_logica": GAMMA_L_LOGICA,
        "rollback_rules": preregistered["rollback_rules"],
    },
    "model": {
        "model_path": MODEL_PATH,
        "base_parameter_hash": base_parameter_hash,
        "verified_file_hashes": verified_model_hashes,
    },
    "z3": {
        "anchor_consistency": anchor_check,
        "sanity_all_passed": bool(all_sanity_passed),
        "sanity_sha256": sha256_file(Z3_SANITY_FILE),
    },
    "base": base_evaluation,
    "fire_test_verdict": verdict_object,
    "runs": all_runs,
}

save_json_atomic(final_result, FINAL_FILE)
final_sha = sha256_file(FINAL_FILE)

manifest = {
    "experiment": EXPERIMENT,
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "hardware": hardware_record,
    "artifacts": {
        "final_results": {"path": FINAL_FILE, "sha256": final_sha},
        "preregistered_criteria": {"path": PREREG_FILE, "sha256": prereg_sha},
        "z3_sanity": {"path": Z3_SANITY_FILE,
                      "sha256": sha256_file(Z3_SANITY_FILE)},
    },
    "fire_test_passed": bool(fire_test_passed),
}
save_json_atomic(manifest, MANIFEST_FILE)

print("\n" + "=" * 70)
print("EXP05 — BEATRIZ FIRE TEST TERMINADO")
print("=" * 70)
print("Final SHA256:", final_sha)
print("Prereg SHA256:", prereg_sha)
print("Manifest SHA256:", sha256_file(MANIFEST_FILE))
print("\nVeredicto por semilla:")
print(json.dumps(criteria_results, indent=2))
print(f"\nBEATRIZ pasa todo en {beatriz_pass_count}/3 semillas")
print(f"NONE falla en {none_fail_count}/3 semillas")
print("\n" + ("*" * 70))
if fire_test_passed:
    print("*** FIRE TEST: PASSED ***")
    print("*** La dinamica del Manual queda validada con datos ficticios.")
    print("*** Siguiente fase: experimento con datos reales.")
else:
    print("*** FIRE TEST: NOT PASSED ***")
    print("*** Revisar criterios por semilla antes de continuar.")
print("*" * 70)
print("\nRECORDATORIO: descarga los JSON de /kaggle/working AHORA.")


/usr/local/lib/python3.12/dist-packages/z3/z3core.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Experiment: EXP05 — Beatriz Fire Test
Device: cuda
{
  "device": "cuda",
  "cuda_available": true,
  "gpu_count": 2,
  "gpu_names": [
    "Tesla T4",
    "Tesla T4"
  ],
  "gpu_used": "cuda:0 only",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "z3_version": "4.12.2",
  "cublas_workspace_config": ":4096:8"
}
Model path: /kaggle/input/datasets/rahulbhat44/gpt-2-offline-model-and-tokenizer-for-kaggle/gpt2_model/gpt2_model
Hash verified: model.safetensors
Hash verified: config.json
Hash verified: vocab.json
Hash verified: merges.txt

FASE 0A: VERIFICACION Z3 DE CONSISTENCIA DEL CORPUS ANCLA
{
  "consistent": true,
  "z3_result": "sat",
  "n_axioms": 24,
  "check_seconds": 0.0071048736572265625
}
>>> Ancla consistente. Entrenamiento autorizado.

FASE 0B: EQUIVALENCIA Z3 <-> LOOKUP
Seed 11: 672 claims | mismatches: 0 | z3: 7.98 ms/claim | PASS
Seed 22: 672 claims | mismatches: 0 | z3: 7.90 ms/claim | PASS
Seed 33: 672 claims | mismatches: 0 | z3: 7.81 ms/claim | PASS
Sanity

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Base truth margin: -0.02799367904663086
Base neutral logprob: -4.39112667242686

STARTING SEED: 11

--- Training NONE ---


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

Seed 22 | BEATRIZ | Epoch 1/8 | P_LIE 0.50 | tm 3.49 | nd 0.135 | triggers: none
Seed 22 | BEATRIZ | Epoch 2/8 | P_LIE 0.50 | tm 8.03 | nd 0.134 | triggers: none
Seed 22 | BEATRIZ | Epoch 3/8 | P_LIE 0.50 | tm 18.23 | nd -0.157 | triggers: none
Seed 22 | BEATRIZ | Epoch 4/8 | P_LIE 0.50 | tm 23.74 | nd -0.382 | triggers: ['R1: neutral_delta=-0.3819']
>>> ROLLBACK en BEATRIZ seed 22: restaurado a epoch 3, rama congelada.

STARTING SEED: 33

--- Training NONE ---


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Seed 33 | NONE | Epoch 1/8 | P_LIE 0.50 | tm -0.69 | nd 0.101 | triggers: ['R3: unknown_delta=8.3261']
>>> ROLLBACK en NONE seed 33: restaurado a epoch 0, rama congelada.

--- Training BEATRIZ ---


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Seed 33 | BEATRIZ | Epoch 1/8 | P_LIE 0.50 | tm 2.54 | nd 0.144 | triggers: none
Seed 33 | BEATRIZ | Epoch 2/8 | P_LIE 0.50 | tm 5.74 | nd 0.034 | triggers: none
Seed 33 | BEATRIZ | Epoch 3/8 | P_LIE 0.50 | tm 18.53 | nd -0.189 | triggers: none
Seed 33 | BEATRIZ | Epoch 4/8 | P_LIE 0.50 | tm 22.21 | nd -0.195 | triggers: none
Seed 33 | BEATRIZ | Epoch 5/8 | P_LIE 0.55 | tm 22.15 | nd -0.134 | triggers: none
Seed 33 | BEATRIZ | Epoch 6/8 | P_LIE 0.60 | tm 24.60 | nd -0.120 | triggers: none
Seed 33 | BEATRIZ | Epoch 7/8 | P_LIE 0.65 | tm 26.56 | nd -0.216 | triggers: none
Seed 33 | BEATRIZ | Epoch 8/8 | P_LIE 0.70 | tm 26.75 | nd -0.208 | triggers: none

EXP05 — BEATRIZ FIRE TEST TERMINADO
Final SHA256: 97359f0464d7a7f7c33a87bca8a3d9e66b212495f5152c4603105ee1493626c1
Prereg SHA256: 57691ac08801e38c63b346579f22091dd0e5da2b7c10ded63a909c678be69f9e
Manifest SHA256: c1e6d68ba7b290a2f34a54b9ccb8d4c043115cc1a1ba3d271fab90aa70564663

Veredicto por semilla:
[
  {
    "seed": 11,
    "beatriz_che